In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

BASE_URL = "http://localhost:3000"
SCRIPT_NAME = "ubo-registration"

LOGIN_EMAIL    = "mathews0912.camargo@hotmail.com"
LOGIN_PASSWORD = "Teste@123"

_auth = requests.post(
    f"{BASE_URL}/auth/login",
    json={"email": LOGIN_EMAIL, "password": LOGIN_PASSWORD},
    headers={"Content-Type": "application/json"},
)
_auth.raise_for_status()

HEADERS = {
    "Authorization": f"Bearer {_auth.json()['token']}",
    "Content-Type": "application/json",
}

print("Autenticado com sucesso.")

In [ ]:
input_path = os.path.join("..", "input", "ubo_registration_documentos.json")

with open(input_path, "r", encoding="utf-8") as f:
    cnpj_list = json.load(f)

print(f"Total de documentos a processar: {len(cnpj_list)}")

In [ ]:
CACHE_DIR = os.path.join("..", "responses", SCRIPT_NAME, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def normalize_doc(document):
    return re.sub(r"\D", "", document)


def clean_doc(doc):
    return doc.replace(".", "").replace("/", "").replace("-", "").strip()


def get_main_address(addresses):
    if not addresses:
        return "N/D"
    for addr in addresses:
        if addr.get("main") and addr.get("active", True):
            return addr.get("address", "N/D")
    for addr in addresses:
        if addr.get("active", True):
            return addr.get("address", "N/D")
    return addresses[0].get("address", "N/D")


def fmt_phone(phones):
    if not phones:
        return "N/D"
    phone = str(phones[0].get("phone", "")).strip()
    return phone if phone else "N/D"


print("Funções auxiliares carregadas. Cache em:", CACHE_DIR)

In [ ]:
WEBHOOK     = "https://webhook.site/c1a70033-31b9-4879-a4e2-6f7ffbb7bd40"
COST_CENTER = 1


def fetch_dados_gerais(document, max_retries=15, wait_sec=2):
    doc_key = normalize_doc(document)
    cache_file = os.path.join(CACHE_DIR, f"{doc_key}.json")

    if os.path.exists(cache_file):
        print(f"  -> Cache encontrado, pulando chamada à API")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    body = {
        "document":    doc_key,
        "cost_center": COST_CENTER,
        "webhook_url": WEBHOOK,
    }
    try:
        r = requests.post(f"{BASE_URL}/dados-gerais", headers=HEADERS, json=body)
        r.raise_for_status()
        order_id = r.json()["order_id"]
    except Exception as e:
        print(f"  Erro ao criar ordem [{document}]: {e}")
        return None

    for _ in range(max_retries):
        try:
            r = requests.get(f"{BASE_URL}/dados-gerais/{order_id}", headers=HEADERS)
            r.raise_for_status()
            result = r.json()
            if result.get("status") == "SUCESSO":
                with open(cache_file, "w", encoding="utf-8") as f:
                    json.dump(result, f, ensure_ascii=False, indent=2)
                return result
        except Exception as e:
            print(f"  Erro ao buscar ordem {order_id} [{document}]: {e}")
            return None
        time.sleep(wait_sec)

    print(f"  Timeout aguardando resultado para {document} (order_id={order_id})")
    return None


print("Função de consulta carregada.")

In [ ]:
def process_cnpj(cnpj, nome, errors):
    rows = []
    doc = normalize_doc(cnpj)

    print(f"  Consultando dados gerais da empresa...")
    reg = fetch_dados_gerais(doc)
    if reg is None:
        errors.append((cnpj, nome, "Erro em /dados-gerais"))
        return rows

    dp = reg.get("data", {}).get("dados_pessoais", {})
    if not dp:
        errors.append((cnpj, nome, "Resposta sem dados_pessoais"))
        return rows

    biz_cnpj = dp.get("document", "N/D")
    biz_name = dp.get("name", "N/D")
    biz_addr = get_main_address(dp.get("address", []))
    biz_stat = dp.get("fiscal_situation", "N/D")
    biz_type = dp.get("legal_nature", "N/D")

    partners  = reg.get("data", {}).get("sociedades", [])
    ubo_names = ", ".join(p.get("name", "") for p in partners) if partners else "N/D"

    if not partners:
        rows.append({
            "Nome":                nome,
            "Business CNPJ":       biz_cnpj,
            "Business Name":       biz_name,
            "Business Address":    biz_addr,
            "Registration Status": biz_stat,
            "Entity Type":         biz_type,
            "List of UBOs":        "N/D",
            "UBOs Ownership (%)":  "N/D",
            "UBO Name":            "N/D",
            "UBO Tax ID":          "N/D",
            "UBO Address":         "N/D",
            "Tax Status":          "N/D",
            "Phone":               "N/D",
            "Date of Birth":       "N/D",
        })
        return rows

    for partner in partners:
        ubo_raw_doc = partner.get("document", "")
        ubo_doc     = normalize_doc(ubo_raw_doc)
        ubo_name    = partner.get("name", "N/D")
        ubo_own     = partner.get("participation", partner.get("relation", "N/D"))

        rows.append({
            "Nome":                nome,
            "Business CNPJ":       biz_cnpj,
            "Business Name":       biz_name,
            "Business Address":    biz_addr,
            "Registration Status": biz_stat,
            "Entity Type":         biz_type,
            "List of UBOs":        ubo_names,
            "UBOs Ownership (%)":  ubo_own,
            "UBO Name":            ubo_name,
            "UBO Tax ID":          ubo_doc or "N/D",
            "UBO Address":         "N/D",
            "Tax Status":          "N/D",
            "Phone":               "N/D",
            "Date of Birth":       "N/D",
        })

    return rows


print("Função de processamento carregada.")

In [ ]:
all_rows = []
errors   = []

for entry in cnpj_list:
    nome = entry.get("nome", "")
    cnpj = entry.get("documento", "")
    print(f"\nProcessando: {nome} ({cnpj})")
    all_rows.extend(process_cnpj(cnpj, nome, errors))

output_dir = os.path.join("..", "responses", SCRIPT_NAME)
os.makedirs(output_dir, exist_ok=True)
erros_path = os.path.join(output_dir, "erros_registro.txt")

with open(erros_path, "w", encoding="utf-8") as f:
    f.write(f"Erros de consulta — {len(errors)} ocorrência(s)\n")
    f.write("=" * 50 + "\n\n")
    for doc, nome_err, reason in errors:
        f.write(f"{nome_err} | {doc} | {reason}\n")

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO")
print(f"{'='*50}")
print(f"Total de documentos buscados : {len(cnpj_list)}")
print(f"Linhas geradas               : {len(all_rows)}")
print(f"Erros registrados            : {len(errors)}")
if errors:
    print(f"\nArquivo de erros salvo em: {erros_path}")

In [ ]:
ALL_COLS = [
    "Nome",
    "Business CNPJ",
    "Business Name",
    "Business Address",
    "Registration Status",
    "Entity Type",
    "List of UBOs",
    "UBOs Ownership (%)",
    "UBO Name",
    "UBO Tax ID",
    "UBO Address",
    "Tax Status",
    "Phone",
    "Date of Birth",
]

# Cores pastéis por grupo de coluna: (header_hex, data_hex)
# Amarelo  → identidade do input (Nome)
# Azul     → dados da empresa (CNPJ, Name, Address)
# Verde    → situação legal (Registration Status, Entity Type)
# Laranja  → composição societária (List of UBOs, Ownership)
# Roxo     → sócio individual (UBO Name, Tax ID)
# Cinza    → dados pendentes/não coletados (Address, Tax Status, Phone, DOB)
COLORS_UBO = {
    "Nome":                ("FFD966", "FFFCE8"),
    "Business CNPJ":       ("9DC3E6", "EBF3FB"),
    "Business Name":       ("9DC3E6", "EBF3FB"),
    "Business Address":    ("9DC3E6", "EBF3FB"),
    "Registration Status": ("A9D18E", "EEF5E9"),
    "Entity Type":         ("A9D18E", "EEF5E9"),
    "List of UBOs":        ("F4B183", "FEF3EA"),
    "UBOs Ownership (%)":  ("F4B183", "FEF3EA"),
    "UBO Name":            ("C9A0DC", "F5EEF8"),
    "UBO Tax ID":          ("C9A0DC", "F5EEF8"),
    "UBO Address":         ("C0C0C0", "F5F5F5"),
    "Tax Status":          ("C0C0C0", "F5F5F5"),
    "Phone":               ("C0C0C0", "F5F5F5"),
    "Date of Birth":       ("C0C0C0", "F5F5F5"),
}

THIN_BORDER = Border(
    left=Side(style="thin", color="D0D0D0"),
    right=Side(style="thin", color="D0D0D0"),
    top=Side(style="thin", color="D0D0D0"),
    bottom=Side(style="thin", color="D0D0D0"),
)


def format_sheet(ws, df, col_colors):
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    data_align = Alignment(horizontal="left", vertical="center", wrap_text=False)

    for col_idx, col_name in enumerate(df.columns, start=1):
        header_hex, data_hex = col_colors.get(col_name, ("D9D9D9", "F5F5F5"))
        header_fill = PatternFill("solid", fgColor=header_hex)
        data_fill = PatternFill("solid", fgColor=data_hex)

        header_cell = ws.cell(row=1, column=col_idx)
        header_cell.fill = header_fill
        header_cell.font = Font(bold=True, color="3B3B3B", size=10)
        header_cell.alignment = header_align
        header_cell.border = THIN_BORDER

        for row_idx in range(2, ws.max_row + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.fill = data_fill
            cell.border = THIN_BORDER
            cell.font = Font(size=10)
            cell.alignment = data_align

    ws.row_dimensions[1].height = 32
    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = get_column_letter(col_idx)
        max_content = max(
            len(str(col_name)),
            max(
                (len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(2, ws.max_row + 1)),
                default=0,
            ),
        )
        ws.column_dimensions[col_letter].width = min(max_content + 3, 55)

    ws.freeze_panes = "A2"


df_ubo = pd.DataFrame(all_rows, columns=ALL_COLS) if all_rows else pd.DataFrame(columns=ALL_COLS)

output_file = os.path.join(output_dir, "relatorio_ubo.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_ubo.to_excel(writer, sheet_name="Relatório UBO", index=False)
    format_sheet(writer.sheets["Relatório UBO"], df_ubo, COLORS_UBO)

print(f"Arquivo '{output_file}' gerado com sucesso.")
print(f"  Total de linhas de dados: {len(all_rows)}")